In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import random
import sys
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../../../atlas_extension/mapqc/")
import mapqc
import plotting

In [ ]:
%load_ext lab_black

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
atlas_path = "../../data/HLCA_full_v1.1_emb.h5ad"

In [ ]:
adata_full = sc.read_h5ad(atlas_path)

subset to query batch of choice:

In [ ]:
query_batch = "Banovich_Kropski_2020"

In [ ]:
adata = adata_full[
    (adata_full.obs.core_or_extension == "core")
    | (adata_full.obs.dataset == query_batch)
].copy()

add precalculated umap coordinates:

In [ ]:
umap_coords = pd.read_csv(f"./umaps/{query_batch}_umap.csv", index_col=0)

In [ ]:
adata.obsm["X_umap"] = umap_coords.loc[adata.obs.index, :].values

### Prepare adata:

In [ ]:
ref_q_key = "core_or_extension"
batch_key = "dataset"
ref_cat = "core"
query_cat = "extension"
condition_key = "lung_condition"

In [ ]:
if query_batch in (adata.obs.loc[adata.obs[ref_q_key] == ref_cat, batch_key].values):
    print(f"Setting batch {query_batch} to 'query' as query simulation.")
    adata.obs[ref_q_key] = adata.obs[ref_q_key].tolist()
    adata.obs.loc[adata.obs[batch_key] == query_batch, ref_q_key] = query_cat

In [ ]:
adata.obs.loc[adata.obs.dataset == query_batch, condition_key].unique()

In [ ]:
condition_of_interest = "Healthy"
print(f"Subsetting query to condition {condition_of_interest}")
adata_healthy = adata[
    (adata.obs[ref_q_key] == ref_cat)
    | (adata.obs[condition_key] == condition_of_interest)
].copy()

In [ ]:
condition_of_interest = "IPF"
print(f"Subsetting query to condition {condition_of_interest}")
adata_disease = adata[
    (adata.obs[ref_q_key] == ref_cat)
    | (adata.obs[condition_key] == condition_of_interest)
].copy()

### Run mapQC on healthy:

In [ ]:
n_nhoods = 500
k = 1000

In [ ]:
query_cell_idc_healthy = np.where(adata_healthy.obs[ref_q_key] == query_cat)[0].tolist()
random.seed(0)
center_cell_idc_healthy = random.sample(query_cell_idc_healthy, k=n_nhoods)
center_cells_healthy = np.array(
    adata_healthy.obs.iloc[center_cell_idc_healthy, :].index
)

In [ ]:
results_healthy = mapqc.mapQC_results(
    adata=adata_healthy,
    center_cells=center_cells_healthy,
    k=k,
    ref_q_key="core_or_extension",
    ref_cat="core",
    query_cat="extension",
    batch_key="dataset",
    exclude_same_batch_for_ref_samples=True,
)

In [ ]:
results_healthy.calculate_sample_dists_to_ref_samples()

In [ ]:
results_healthy.calculate_nhood_statistics()

In [ ]:
results_healthy.nhood_stats

In [ ]:
# temp = results_healthy.adata[results_healthy.adata.obs.core_or_extension != "core", :]
# sc.pp.neighbors(temp, n_neighbors=30)
# sc.tl.umap(temp)
# sc.pl.umap(
#     temp,
#     color="core_or_extension",
#     palette={
#         results_healthy.ref_cat: "tab:orange",
#         results_healthy.query_cat: "tab:blue",
#     },
#     title=f"{query_batch} (disease)",
#     frameon=False,
# )

In [ ]:
# sc.pl.umap(
#     results_healthy.adata[results_healthy.adata.obs.core_or_extension == "core", :],
#     color="core_or_extension",
#     palette={
#         results_healthy.ref_cat: "tab:orange",
#         results_healthy.query_cat: "tab:blue",
#     },
#     title=f"{query_batch} (disease)",
#     frameon=False,
# )

### Plot:

In [ ]:
sc.set_figure_params(figsize=(5, 5))

#### healthy:

In [ ]:
sc.pl.umap(
    results_healthy.adata,
    color="core_or_extension",
    palette={
        results_healthy.ref_cat: "tab:orange",
        results_healthy.query_cat: "tab:blue",
    },
    title=f"{query_batch} (healthy)",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    results_healthy.adata,
    color="ann_level_3",
    # title=f"{query_batch} (disease)",
    frameon=False,
)

In [ ]:
plotting.plot_umap_significance(results_healthy)

In [ ]:
plotting.plot_umap_zscored_q_dist_to_ref(results_healthy)

In [ ]:
plotting.plot_umap_q_ref_sample_shaded(results_healthy)

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(results_healthy)

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(
    results_healthy, use_zscored_dists=True
)

#### Aggregate according to pre-determined partitioning (e.g. cell type)

In [ ]:
partitioning_key = "ann_level_3"
stats_df_healthy = results_healthy.nhood_stats.copy()
stats_df_healthy[partitioning_key] = results_healthy.adata.obs.loc[
    stats_df_healthy.index, partitioning_key
]
stats_df_healthy["-log10(pval_corr)"] = -np.log10(
    stats_df_healthy.pval_corr.values.astype(float)
)
fig = plt.figure(figsize=(9, 3))
ax = fig.add_subplot()
sns.boxplot(
    stats_df_healthy, x=partitioning_key, y="-log10(pval_corr)", ax=ax, saturation=0.4
)
sns.stripplot(
    stats_df_healthy,
    x=partitioning_key,
    y="-log10(pval_corr)",
    dodge=True,
    ax=ax,
    size=3,
    palette="dark:black",
)
plt.axhline(-np.log10(0.05), linestyle="--")
ax.tick_params(axis="x", rotation=90)
ax.set_xlabel("Cell type")

#### zoom in on neighborhood in cell type of interest:

In [ ]:
results_healthy.nhood_stats["cell_type"] = results_healthy.adata.obs.loc[
    results_healthy.nhood_stats.index, "ann_level_3"
]

In [ ]:
ctoi_1 = "AT2"
nhood_of_int_healthy = results_healthy.nhood_stats.index[
    np.where(results_healthy.nhood_stats.cell_type == ctoi_1)[0][0]
]

In [ ]:
plotting.plot_umap_neighborhood(
    results_healthy, neighborhood=nhood_of_int_healthy, scale=0.5
)

In [ ]:
plotting.plot_umap_q_ref_sample_shaded(
    results_healthy,
    neighborhood=nhood_of_int_healthy,
    include_legend=True,
    reference_only=True,
    seed=0,
    dotsize=100,
)

In [ ]:
plotting.plot_umap_q_ref_sample_shaded(
    results_healthy,
    neighborhood=nhood_of_int_healthy,
    include_legend=True,
    seed=0,
    dotsize=100,
)

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(
    results_healthy, neighborhood=nhood_of_int_healthy
)

In [ ]:
results_healthy.nhood_stats.loc[nhood_of_int_healthy, :]

#### Zoom in on significantly differentially mixing nhood:

In [ ]:
nhood_of_int_healthy_sign = results_healthy.nhood_stats.sort_values(
    by="pval_corr"
).index[0]

In [ ]:
plotting.plot_umap_neighborhood(
    results_healthy, neighborhood=nhood_of_int_healthy_sign, scale=0.5
)

In [ ]:
plotting.plot_umap_q_ref_sample_shaded(
    results_healthy,
    neighborhood=nhood_of_int_healthy_sign,
    include_legend=True,
    seed=0,
    dotsize=100,
)

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(
    results_healthy, neighborhood=nhood_of_int_healthy_sign
)

In [ ]:
results_healthy.nhood_stats.loc[nhood_of_int_healthy_sign, :]

In [ ]:
# nhood_idx = np.where(mapqc_results.center_cells == neighborhood)[0][0]
# nhood_cells = mapqc_results.nhood_cells[nhood_idx, :]
# adata_to_plot = mapqc_results.adata[nhood_cells, :].copy()

In [ ]:
# sc.pl.umap(adata_nhood_healthy)

## Optional: run mapQC on disease

In [ ]:
query_cell_idc_disease = np.where(adata_disease.obs[ref_q_key] == query_cat)[0].tolist()
random.seed(0)
center_cell_idc_disease = random.sample(query_cell_idc_disease, k=n_nhoods)
center_cells_disease = np.array(
    adata_disease.obs.iloc[center_cell_idc_disease, :].index
)

In [ ]:
results_disease = mapqc.mapQC_results(
    adata=adata_disease,
    center_cells=center_cells_disease,
    k=k,
    ref_q_key="core_or_extension",
    ref_cat="core",
    query_cat="extension",
    batch_key="dataset",
)

In [ ]:
results_disease.calculate_sample_dists_to_ref_samples()

In [ ]:
results_disease.calculate_nhood_statistics()

#### Plot disease results:

In [ ]:
sc.pl.umap(
    results_disease.adata,
    color="core_or_extension",
    palette={
        results_disease.ref_cat: "tab:orange",
        results_disease.query_cat: "tab:blue",
    },
    title=f"{query_batch} (disease)",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    results_disease.adata,
    color="ann_level_3",
    # title=f"{query_batch} (disease)",
    frameon=False,
)

In [ ]:
plotting.plot_umap_significance(results_disease)

In [ ]:
plotting.plot_umap_q_ref_sample_shaded(results_healthy)

In [ ]:
# plotting.plot_umap_neighborhood(
#     results_disease, neighborhood="...", scale=0.5
# )

In [ ]:
plotting.plot_umap_zscored_q_dist_to_ref(results_disease)

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(results_disease)

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(
    results_disease, use_zscored_dists=True
)

In [ ]:
plotting.plot_umap_q_ref_sample_shaded(results_disease)

plot specific neighborhood of cell type of interest 1:

In [ ]:
results_disease.nhood_stats["cell_type"] = results_disease.adata.obs.loc[
    results_disease.nhood_stats.index, "ann_level_3"
]

In [ ]:
nhood_of_int_disease = results_disease.nhood_stats.index[
    np.where(results_disease.nhood_stats.cell_type == ctoi_1)[0][0]
]

In [ ]:
plotting.plot_umap_neighborhood(
    results_disease, neighborhood=nhood_of_int_disease, scale=0.5
)

In [ ]:
plotting.plot_umap_q_ref_sample_shaded(
    results_disease, neighborhood=nhood_of_int_disease, include_legend=True
)

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(
    results_disease, neighborhood=nhood_of_int_disease
)

pvalue of neighborhood:

In [ ]:
results_disease.nhood_stats.loc[nhood_of_int_disease, :]

In [ ]:
partitioning_key = "ann_level_3"
stats_df_disease = results_disease.nhood_stats.copy()
stats_df_disease[partitioning_key] = results_disease.adata.obs.loc[
    stats_df_disease.index, partitioning_key
]
stats_df_disease["-log10(pval_corr)"] = -np.log10(
    stats_df_disease.pval_corr.values.astype(float)
)
fig = plt.figure(figsize=(9, 3))
ax = fig.add_subplot()
sns.boxplot(
    stats_df_disease, x=partitioning_key, y="-log10(pval_corr)", ax=ax, saturation=0.4
)
sns.stripplot(
    stats_df_disease,
    x=partitioning_key,
    y="-log10(pval_corr)",
    dodge=True,
    ax=ax,
    size=3,
    palette="dark:black",
)
plt.axhline(-np.log10(0.05), linestyle="--")
ax.tick_params(axis="x", rotation=90)
ax.set_xlabel("Cell type")